# lab 6: rag pe un document propriu

scopul lucrării: să facem un sistem simplu de tip rag (retrieval augmented generation), care răspunde la întrebări folosind un articol mare de fizică atomică.

documentul: M. S. Safronova și alții, *Search for New Physics with Atoms and Molecules*, Reviews of Modern Physics 90, 025008 (2018)

cum lucrează rag:

1. tăiem documentul în bucăți mici
2. transformăm fiecare bucată într-un vector (embedding)
3. pentru o întrebare găsim bucățile cele mai apropiate
4. dăm aceste bucăți unui model de limbaj, împreună cu întrebarea
5. modelul răspunde folosind textul găsit, nu doar ce ține minte

In [1]:
%pip install -q pypdf sentence-transformers transformers accelerate

Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
import numpy as np
import torch
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
print(device)

cuda


## 1. citim documentul

în colab încărcăm întâi fișierul pdf în folderul `lab6_data`.

In [3]:
PDF = "lab6_data/safronova_2018_new_physics_atoms_molecules.pdf"

reader = PdfReader(PDF)
pages = [p.extract_text() or "" for p in reader.pages]
print("pagini:", len(pages))
print("caractere:", sum(len(p) for p in pages))
print(pages[0][:500])

pagini: 121
caractere: 657340
Search for New Physics with Atoms and Molecules
M.S. Safronova 1,2, D. Budker 3,4,5, D. DeMille 6, Derek F. Jackson Kimball 7, A. Derevianko 8 and Charles W.
Clark2
1University of Delaware, Newark, Delaware, USA,
2Joint Quantum Institute, National Institute of Standards and Technology and the University of Maryland,
College Park, Maryland, USA,
3Helmholtz Institute, Johannes Gutenberg University, Mainz, Germany,
4University of California, Berkeley, California, USA,
5Nuclear Science Division, Law


## 2. taiem textul în bucăți

curățăm spațiile în plus. apoi facem bucăți de câte 150 de cuvinte. bucățile se suprapun cu 30 de cuvinte, ca să nu tăiem o idee la mijloc.

pentru fiecare bucată ținem minte și pagina, ca să putem arăta sursa.

de la pagina 92 până la final este doar lista de referințe. acolo sunt multe cuvinte cheie, dar nu sunt explicații. dacă le lăsăm, căutarea găsește des referințe în loc de text util. de aceea luăm doar paginile 1 până la 91.

In [4]:
SIZE, OVERLAP = 150, 30
LAST_PAGE = 91                                # references start at page 92

chunks = []
for page_no, text in enumerate(pages[:LAST_PAGE], start=1):
    words = re.sub(r"\s+", " ", text).split()
    for i in range(0, len(words), SIZE - OVERLAP):
        piece = " ".join(words[i:i + SIZE])
        if len(piece) > 200:                    # skip tiny pieces
            chunks.append({"page": page_no, "text": piece})

print("bucăți:", len(chunks))
print(chunks[10])

bucăți: 657
{'page': 2, 'text': '91 5. Symbols and abbreviations 91 References 91 I. INTRODUCTION A. Recent advances in AMO physics The past two decades have been a transformational era for atomic, molecular and optical (AMO) physics, due to extraordinary accomplishments in the control of matter and light. Experimental breakthroughs, includ- ing laser cooling and trapping of atoms, attainment of Bose-Einstein condensation, optical frequency combs and quantum control - subject of Nobel Prizes in Physics in 1997 (Chu, 1998; Cohen-Tannoudji, 1998; Phillips, 1998), 2001 (Cornell and Wieman, 2002; Ketterle, 2002), 2005 (Glauber, 2006; Hall, 2006; H¨ ansch, 2006), and 2012 (Haroche, 2013; Wineland, 2013), respectively - have led to widespread availability of ultracold (temper- ature T < 1 µK) ions, atoms and molecules, subject to precise interrogation and control. Revolutionary devel- opments on several fronts have been made possible by these advances, aided by improvements in precision time

## 3. embeddings

folosim un model mic, `all-MiniLM-L6-v2`. el face din fiecare bucată un vector de 384 de numere. textele cu sens apropiat au vectori apropiați.

normalizăm vectorii. așa produsul scalar este chiar similaritatea cosinus.

In [5]:
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
emb = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True,
                      batch_size=64, show_progress_bar=True)
print(emb.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

(657, 384)


## 4. căutarea

transformăm întrebarea în vector și luăm bucățile cu similaritatea cea mai mare.

In [6]:
def search(question, k=4):
    q = embedder.encode([question], normalize_embeddings=True)[0]
    sims = emb @ q
    best = np.argsort(-sims)[:k]
    return [(chunks[i], float(sims[i])) for i in best]

for c, s in search("How can atomic clocks be used to search for dark matter?", k=3):
    print(f"pagina {c['page']}, similaritate {s:.3f}")
    print(c["text"][:300], "...\n")

pagina 67, similaritate 0.655
67 in Sec. VII.H). Consequently, atomic physics techniques can be employed to search for dark matter scalar bosons as discussed in detail in Sec. IX. There are also a num- ber of new proposals on the horizon that promise im- proved sensitivity to spin-independent interactions at various length scale ...

pagina 70, similaritate 0.624
virtual exotic particles. Such experiments include the “ﬁfth-force” searches and exper- iments looking for exotic spin-dependent interactions or modiﬁcation of fundamental constants in the presence of massive and/or spin-polarized objects that presumably act as sources of the virtual exotic particle ...

pagina 74, similaritate 0.616
themselves as “glitches” of atomic clocks, for example those onboard satellites of Global Positioning System (Derevianko and Pospelov, 2014): if particular interactions exist, the clocks would become desynchronized as they are swept by a DM ob- ject (Fig. 21). The glitches would propagate through 

## 5. modelul de limbaj

folosim un model mic cu instrucțiuni, `Qwen2.5-1.5B-Instruct`. el încape pe un gpu mic, iar pe cpu merge mai încet.

In [7]:
LLM = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(LLM)
llm = AutoModelForCausalLM.from_pretrained(
    LLM, torch_dtype=torch.float16 if device == "cuda" else torch.float32).to(device)
llm.eval()

def chat(messages, max_new_tokens=250):
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = llm.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0, ids.input_ids.shape[1]:], skip_special_tokens=True)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

## 6. răspunsul cu rag

punem bucățile găsite în prompt. îi cerem modelului să folosească doar acest context și să spună "I don't know" dacă răspunsul nu este acolo.

lucrăm în engleză, pentru că articolul este în engleză.

In [8]:
def ask_rag(question, k=4):
    found = search(question, k)
    context = "\n\n".join(f"[page {c['page']}] {c['text']}" for c, _ in found)
    messages = [
        {"role": "system", "content": "Answer only from the context. "
         "If the answer is not in the context, say: I don't know. Keep it short."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    answer = chat(messages)
    pages_used = sorted({c["page"] for c, _ in found})
    return answer, pages_used

def ask_plain(question):
    return chat([{"role": "user", "content": question + " Keep it short."}])

In [9]:
q = "Which atom gave the most precise measurement of atomic parity violation?"
answer, src = ask_rag(q)
print("ÎNTREBARE:", q)
print("RĂSPUNS RAG:", answer)
print("PAGINI:", src)

ÎNTREBARE: Which atom gave the most precise measurement of atomic parity violation?
RĂSPUNS RAG: The most precise measurement of atomic parity violation was made in 133Cs, according to the given text.
PAGINI: [3, 25, 27]


## 7. cu rag și fără rag

punem aceleași întrebări modelului singur și modelului cu rag. vedem dacă răspunsul se schimbă.

In [10]:
questions = [
    "What is the electron electric dipole moment and why do we search for it?",
    "How can atomic clocks be used to search for dark matter?",
    "Which molecule was used in the ACME experiment?",
    "What is a fifth force and how do atoms help to search for it?",
]

for q in questions:
    answer, src = ask_rag(q)
    print("ÎNTREBARE:", q)
    print("\nFĂRĂ RAG:", ask_plain(q))
    print("\nCU RAG:", answer)
    print("PAGINI:", src)
    print("\n" + "=" * 80 + "\n")

ÎNTREBARE: What is the electron electric dipole moment and why do we search for it?



FĂRĂ RAG: The electron's electric dipole moment measures its overall charge distribution. We seek it to test fundamental symmetries in physics.

CU RAG: The electron electric dipole moment (EDM) is a fundamental property of electrons that measures their intrinsic angular momentum. We search for it because if the electron had a nonzero EDM, it would imply that the universe contains more matter than antimatter, which contradicts our current understanding of the laws of physics. The discovery of a nonzero EDM would provide strong evidence for the existence of additional particles beyond those predicted by the Standard Model of particle physics.
PAGINI: [34, 46, 62, 71]




ÎNTREBARE: How can atomic clocks be used to search for dark matter?



FĂRĂ RAG: Atomic clocks measure extremely precise time intervals, which could potentially help detect gravitational waves or other effects related to dark matter through their influence on cosmic structures.

CU RAG: Atomic clocks can be used to search for dark matter by detecting "glitches" caused when interacting with dark matter objects. These glitches occur due to the desynchronization of the clocks as they pass through regions where dark matter interacts strongly.
PAGINI: [63, 67, 70, 74]




ÎNTREBARE: Which molecule was used in the ACME experiment?



FĂRĂ RAG: The ACME (Antimatter Candidate Experiment) experiment used antihydrogen atoms as its target particles.

CU RAG: ThO
PAGINI: [38, 39, 42, 88]




ÎNTREBARE: What is a fifth force and how do atoms help to search for it?



FĂRĂ RAG: A fifth force refers to an unknown interaction that does not fit into the four known fundamental forces of nature (gravity, electromagnetism, strong nuclear force, and weak nuclear force). Atoms can be used in experiments designed to detect hypothetical particles or fields that might mediate such a fifth force.

CU RAG: A fifth force is a hypothetical type of force beyond the known four fundamental forces of nature (gravity, electromagnetism, strong nuclear force, and weak nuclear force). To search for a fifth force, scientists study its effects on atoms and their constituent particles like electrons, protons, and neutrons. Specifically, they look at how these particles behave when subjected to various conditions, such as different magnetic fields or distances from massive objects. The behavior of atomic transitions and spectra can provide clues about the presence or absence of such forces.
PAGINI: [1, 49, 76, 80]




## 8. o întrebare care nu este în document

un sistem rag bun trebuie să spună că nu știe, nu să inventeze.

In [11]:
q = "What is the best recipe for chocolate cake?"
answer, src = ask_rag(q)
print("RĂSPUNS RAG:", answer)
print("cea mai mare similaritate:", round(search(q, 1)[0][1], 3))

RĂSPUNS RAG: I don't know.
cea mai mare similaritate: 0.174


## 9. întrebare în limba română

modelul de embeddings este antrenat pe engleză. să vedem ce se întâmplă cu o întrebare în română.

In [12]:
q = "Cum pot fi folosite ceasurile atomice pentru a căuta materia întunecată?"
for c, s in search(q, 3):
    print(f"pagina {c['page']}, similaritate {s:.3f}")

pagina 88, similaritate 0.297
pagina 91, similaritate 0.242
pagina 79, similaritate 0.240


## concluzii

* documentul are 121 de pagini. după ce am scos referințele au rămas 657 de bucăți, fiecare cu un vector de 384 de numere.
* tăierea referințelor a contat mult. înainte, la întrebarea despre ceasurile atomice, căutarea găsea paginile 97 și 99 (referințe), iar răspunsul era greșit (vorbea despre gravitație). după tăiere găsește paginile 63 până la 74, iar răspunsul este corect: ceasurile au "glitch"-uri când trec prin obiecte de materie întunecată.
* la întrebările cu un fapt precis rag ajută clar. modelul singur a spus că în experimentul ACME s-au folosit atomi de antihidrogen, ceea ce este inventat. cu rag a răspuns corect: molecula ThO. la fel, cu rag a găsit că cea mai precisă măsurare a violării parității este în 133Cs.
* rag nu rezolvă tot. la întrebarea despre momentul de dipol electric al electronului modelul mic a amestecat lucruri: a spus greșit că este legat de momentul cinetic. un model mic poate citi greșit contextul.
* la o întrebare care nu are legătură cu documentul (rețeta de tort) similaritatea maximă a fost doar 0.174, iar modelul a spus corect "I don't know".
* la întrebarea în română similaritățile au fost mici (sub 0.3), pentru că modelul de embeddings este antrenat pe engleză. pentru întrebări în română ar trebui un model multilingv, de exemplu paraphrase-multilingual-MiniLM-L12-v2.
* concluzia: rag face modelul să răspundă din document și să arate paginile, deci putem verifica răspunsul. calitatea depinde mult de ce bucăți găsește căutarea.